# Module 2 --- Functions, Classes and PyTorch `nn.Module`

**Course:** Python and Machine Learning with PyTorch
**Companion slides:** `Presentation_2_Classes_Modules.pdf`

---

### What you will do here

1. **Setup** --- libraries, imports, device.
2. **Theory in practice** --- scope, lambdas, generators, then plain Python
   classes, inheritance, and finally `MyFirstNet(nn.Module)` with a real
   training loop.
3. **Challenge** --- make the network depth a constructor argument using
   `nn.ModuleList`.

### The one idea to hold onto

A PyTorch model **is a Python class**. `__init__` declares what the model owns,
`forward` declares what it does. There is no extra magic.

## 1. Setup

In [ ]:
!pip install torch torchvision matplotlib pandas scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification   # datasets only, no models
from sklearn.metrics import accuracy_score         # metrics only

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

## 2. Theory in Practice

### 2.1 Scope, and the mutable default trap

Python resolves names with the **LEGB** rule: Local, Enclosing, Global, Built-in.

In [ ]:
lr = 0.01                      # global

def train():
    lr = 0.1                   # LOCAL, shadows the global
    print("  inside train, lr =", lr)

train()
print("outside, lr =", lr, " <- the global was never touched")


def make_counter():
    count = 0                  # enclosing scope
    def increment():
        nonlocal count         # without this, count would be local to increment
        count += 1
        return count
    return increment

c = make_counter()
print("\ncounter:", c(), c(), c())

In [ ]:
# The mutable default argument trap.

def add_loss_wrong(value, history=[]):
    history.append(value)      # the SAME list is reused on every call
    return history

print("wrong:", add_loss_wrong(1.0))
print("wrong:", add_loss_wrong(2.0), " <- 1.0 is still there")


def add_loss_right(value, history=None):
    if history is None:
        history = []           # a fresh list per call
    history.append(value)
    return history

print("\nright:", add_loss_right(1.0))
print("right:", add_loss_right(2.0))

### 2.2 Lambdas and higher-order functions

In [ ]:
square = lambda x: x ** 2
print("square(4) =", square(4))

pairs = [("resnet", 0.91), ("vgg", 0.87), ("mlp", 0.72), ("knn", 0.79)]

pairs_sorted = sorted(pairs, key=lambda p: p[1], reverse=True)
print("\nsorted by accuracy:", pairs_sorted)

accs = list(map(lambda p: p[1], pairs))
good = list(filter(lambda p: p[1] > 0.8, pairs))
print("map    :", accs)
print("filter :", good)

# A lambda applied to a tensor, the way torchvision transforms use them
flatten = lambda t: t.reshape(t.shape[0], -1)
img_batch = torch.randn(4, 3, 8, 8)
print("\nbefore flatten:", tuple(img_batch.shape))
print("after  flatten:", tuple(flatten(img_batch).shape))

### 2.3 Generators

A generator produces values **lazily**, one at a time, instead of building a
whole list in memory. This is exactly how a `DataLoader` yields batches, which
is the subject of Module 3.

In [ ]:
def batch_indices(n, batch_size, shuffle=False):
    """Yield index chunks without materialising the full list."""
    order = torch.randperm(n) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


print("sequential batches:")
for idx in batch_indices(10, 4):
    print("  ", idx.tolist())

print("\nshuffled batches:")
torch.manual_seed(SEED)
for idx in batch_indices(10, 4, shuffle=True):
    print("  ", idx.tolist())

# A generator is consumed once and then exhausted
gen = batch_indices(6, 3)
print("\nfirst pass :", [i.tolist() for i in gen])
print("second pass:", [i.tolist() for i in gen], " <- empty, already consumed")

In [ ]:
# Memory: a generator expression uses ( ), a list comprehension uses [ ]
import sys

as_list = [x ** 2 for x in range(100000)]
as_gen  = (x ** 2 for x in range(100000))

print("list object size:", sys.getsizeof(as_list), "bytes")
print("generator size  :", sys.getsizeof(as_gen), "bytes")
print("both sum to     :", sum(as_list), "and", sum(x ** 2 for x in range(100000)))

### 2.4 A plain Python class

Two dunder methods to notice: `__len__` and `__getitem__`. They are exactly what
a PyTorch `Dataset` needs, as you will see in Module 3.

In [ ]:
class SimpleDataset:
    """A blueprint. Nothing exists until you instantiate it."""

    def __init__(self, name, samples):     # the CONSTRUCTOR
        self.name = name                   # attributes live on the instance
        self.samples = list(samples)

    def __len__(self):                     # enables len(obj)
        return len(self.samples)

    def __getitem__(self, i):              # enables obj[i] and iteration
        return self.samples[i]

    def __repr__(self):                    # what print(obj) shows
        return f"SimpleDataset(name={self.name!r}, n={len(self)})"

    def mean(self):                        # an ordinary METHOD
        return sum(self.samples) / len(self.samples)


ds = SimpleDataset("iris-sepal", [5.1, 4.9, 4.7, 5.4, 5.0])
print(ds)
print("len      :", len(ds))
print("ds[2]    :", ds[2])
print("mean     :", round(ds.mean(), 3))
print("iterable :", [x for x in ds])

# Two instances, one class, independent state
ds2 = SimpleDataset("other", [1.0, 2.0])
print("\n", ds, "and", ds2, "are independent objects")
print("ds.mean() is really SimpleDataset.mean(ds):", SimpleDataset.mean(ds) == ds.mean())

### 2.5 Inheritance and `super()`

Forgetting `super().__init__()` is the single most common bug when subclassing
`nn.Module`. Practise it here on a plain class first.

In [ ]:
class BaseModel:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Model {self.name}"


class Classifier(BaseModel):                 # inherits from BaseModel
    def __init__(self, name, n_classes):
        super().__init__(name)               # run the PARENT constructor first
        self.n_classes = n_classes

    def describe(self):                      # OVERRIDE, but reuse the parent
        return super().describe() + f" with {self.n_classes} classes"


clf = Classifier("mlp", 10)
print(clf.describe())
print("isinstance of BaseModel:", isinstance(clf, BaseModel))


# What happens when you forget super().__init__()
class Broken(BaseModel):
    def __init__(self, name, n_classes):
        self.n_classes = n_classes           # parent constructor never ran

b = Broken("oops", 3)
try:
    b.describe()
except AttributeError as err:
    print("\nExpected error:", err)

### 2.6 `MyFirstNet`: subclassing `nn.Module`

The exercise announced on the slides.

- `__init__` runs **once** and creates the layers.
- `forward` runs on **every batch** and applies them.
- You call `model(x)`, never `model.forward(x)`.

In [ ]:
class MyFirstNet(nn.Module):
    def __init__(self, in_features=4, hidden=16, n_classes=3):
        super().__init__()                              # MANDATORY first line
        self.fc1  = nn.Linear(in_features, hidden)      # layer DEFINITION
        self.relu = nn.ReLU()
        self.fc2  = nn.Linear(hidden, n_classes)

    def forward(self, x):                               # layer USE
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x                                        # raw scores = logits


model = MyFirstNet(in_features=4, hidden=16, n_classes=3)
print(model)

In [ ]:
# Push a dummy tensor through the network
dummy = torch.randn(8, 4)          # a batch of 8 samples, 4 features each

out = model(dummy)                 # calls forward() under the hood

print("input  shape:", tuple(dummy.shape))
print("output shape:", tuple(out.shape), " -> 8 samples, 3 class scores each")
print("\nlogits (raw scores):\n", out[:3])

probs = torch.softmax(out, dim=1)  # turn scores into probabilities
print("\nprobabilities:\n", probs[:3])
print("row sums (should be 1):", probs.sum(dim=1)[:3])
print("\npredicted classes:", out.argmax(dim=1).tolist())

In [ ]:
# Inspect what nn.Module registered for you
print("--- named parameters ---")
for name, p in model.named_parameters():
    print(f"{name:12s} {str(tuple(p.shape)):10s} requires_grad={p.requires_grad}")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\ntrainable parameters:", n_params, "= 16*4+16 + 3*16+3")

print("\n--- named submodules ---")
for name, module in model.named_children():
    print(f"{name:6s} -> {module}")

# A layer's weight matrix is (out_features, in_features), note the order
print("\nfc1.weight shape:", tuple(model.fc1.weight.shape), "(out, in)")

### 2.7 A real training loop

The five steps from the slides, on data generated with scikit-learn. Note that
scikit-learn only produces the dataset. The model and the optimisation are pure
PyTorch.

In [ ]:
# ---- user-adjustable parameters ------------------------------------------
N_SAMPLES  = 600
N_FEATURES = 4
N_CLASSES  = 3
HIDDEN     = 32
EPOCHS     = 150
LR         = 0.05
# --------------------------------------------------------------------------

X_np, y_np = make_classification(
    n_samples=N_SAMPLES, n_features=N_FEATURES, n_informative=N_FEATURES,
    n_redundant=0, n_classes=N_CLASSES, n_clusters_per_class=1,
    class_sep=1.4, random_state=SEED,
)

X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)      # CrossEntropyLoss wants int64

# Standardise, then split 80 / 20
X = (X - X.mean(dim=0)) / X.std(dim=0)
n_train = int(0.8 * len(X))
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(SEED))
tr, te = perm[:n_train], perm[n_train:]

X_train, y_train = X[tr].to(device), y[tr].to(device)
X_test,  y_test  = X[te].to(device), y[te].to(device)

print("train:", tuple(X_train.shape), "| test:", tuple(X_test.shape))
print("class counts (train):", torch.bincount(y_train).tolist())

In [ ]:
model = MyFirstNet(N_FEATURES, HIDDEN, N_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()                     # expects logits + int labels
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {"loss": [], "train_acc": [], "test_acc": []}

for epoch in range(EPOCHS):
    # ---- train ----------------------------------------------------------
    model.train()
    optimizer.zero_grad()                   # 5. clear old gradients
    logits = model(X_train)                 # 1. forward
    loss = criterion(logits, y_train)       # 2. loss
    loss.backward()                         # 3. backward
    optimizer.step()                        # 4. update

    # ---- evaluate -------------------------------------------------------
    model.eval()
    with torch.no_grad():
        train_acc = (logits.argmax(1) == y_train).float().mean().item()
        test_acc  = (model(X_test).argmax(1) == y_test).float().mean().item()

    history["loss"].append(loss.item())
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)

    if epoch % 25 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} "
              f"| train acc {train_acc:.3f} | test acc {test_acc:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))

axes[0].plot(history["loss"], color="crimson")
axes[0].set_title("training loss"); axes[0].set_xlabel("epoch"); axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["test_acc"], label="test")
axes[1].set_title("accuracy"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0, 1.02)

plt.tight_layout(); plt.show()

In [ ]:
# Final evaluation, and a cross-check against scikit-learn's metric
model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)

torch_acc = (preds == y_test).float().mean().item()
sk_acc = accuracy_score(y_test.cpu().numpy(), preds.cpu().numpy())

print(f"accuracy (torch)   : {torch_acc:.4f}")
print(f"accuracy (sklearn) : {sk_acc:.4f}")

# Confusion matrix computed with pure tensor operations
cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.long)
for t, p in zip(y_test.cpu(), preds.cpu()):
    cm[t, p] += 1

print("\nconfusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm.numpy(),
                   index=[f"true {i}" for i in range(N_CLASSES)],
                   columns=[f"pred {i}" for i in range(N_CLASSES)]))

### 2.8 Saving, loading, and `nn.Sequential`

In [ ]:
# Save only the weights, not the Python object
torch.save(model.state_dict(), "myfirstnet.pt")
print("state_dict keys:", list(model.state_dict().keys()))

# Rebuild the SAME architecture, then load
reloaded = MyFirstNet(N_FEATURES, HIDDEN, N_CLASSES)
reloaded.load_state_dict(torch.load("myfirstnet.pt", map_location="cpu"))
reloaded.eval()

with torch.no_grad():
    same = torch.allclose(reloaded(X_test.cpu()), model(X_test).cpu(), atol=1e-6)
print("reloaded model gives identical outputs:", same)

In [ ]:
# nn.Sequential: a shortcut when the model is a plain stack with no branching
seq = nn.Sequential(
    nn.Linear(N_FEATURES, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, N_CLASSES),
)
print(seq)
print("\nsame output shape:", tuple(seq(torch.randn(8, N_FEATURES)).shape))
print("parameter count  :", sum(p.numel() for p in seq.parameters()))

## 3. Challenge

**Challenge 1.** Write `DeepNet(nn.Module)` whose constructor takes a list of
layer sizes, for example `[4, 32, 32, 16, 3]`, and builds the corresponding
stack. Use `nn.ModuleList` so the parameters are registered. Apply ReLU between
layers but **not** after the last one.

**Challenge 2.** Why does a plain Python list of layers fail? Demonstrate it.

**Challenge 3.** Train `DeepNet([4, 64, 32, 3])` on the data above and compare
its test accuracy to `MyFirstNet`.

In [ ]:
# TODO Challenge 1
class DeepNet(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        # your code here
        raise NotImplementedError

    def forward(self, x):
        # your code here
        raise NotImplementedError

### Solutions

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(a, b) for a, b in zip(sizes[:-1], sizes[1:])]
        )

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:     # no activation after the last layer
                x = torch.relu(x)
        return x


deep = DeepNet([N_FEATURES, 64, 32, N_CLASSES])
print(deep)
print("\noutput shape:", tuple(deep(torch.randn(8, N_FEATURES)).shape))
print("trainable parameters:", sum(p.numel() for p in deep.parameters()))

In [ ]:
# Challenge 2: why a plain list fails.

class BrokenNet(nn.Module):
    def __init__(self, sizes):
        super().__init__()
        self.layers = [nn.Linear(a, b) for a, b in zip(sizes[:-1], sizes[1:])]

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


broken = BrokenNet([4, 16, 3])
good   = DeepNet([4, 16, 3])

print("BrokenNet parameters:", sum(p.numel() for p in broken.parameters()))
print("DeepNet   parameters:", sum(p.numel() for p in good.parameters()))
print("\nA plain list is invisible to .parameters(), so the optimiser would")
print("receive nothing and the layers would never be trained or moved by .to(device).")

In [ ]:
# Challenge 3: train DeepNet and compare.

def train_model(net, epochs=EPOCHS, lr=LR):
    net = net.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        net.train()
        opt.zero_grad()
        loss = crit(net(X_train), y_train)
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        acc = (net(X_test).argmax(1) == y_test).float().mean().item()
    return acc, loss.item()


torch.manual_seed(SEED)
shallow_acc, shallow_loss = train_model(MyFirstNet(N_FEATURES, HIDDEN, N_CLASSES))

torch.manual_seed(SEED)
deep_acc, deep_loss = train_model(DeepNet([N_FEATURES, 64, 32, N_CLASSES]))

print(f"MyFirstNet [4, 32, 3]      -> test acc {shallow_acc:.4f} | loss {shallow_loss:.4f}")
print(f"DeepNet    [4, 64, 32, 3]  -> test acc {deep_acc:.4f} | loss {deep_loss:.4f}")
print("\nOn a dataset this easy, extra depth buys little. More capacity is not")
print("automatically better, which is a lesson worth carrying into Module 4.")

## Recap

| Concept | Key point |
|---|---|
| Scope | LEGB; never use a mutable default argument |
| Generator | `yield` produces values lazily, like a `DataLoader` |
| Class | `__init__` builds state, `self` is the instance |
| Dunder methods | `__len__` and `__getitem__` reappear in Module 3 |
| `super().__init__()` | Mandatory in every `nn.Module` subclass |
| `__init__` vs `forward` | Declare layers vs apply them |
| Calling a model | `model(x)`, never `model.forward(x)` |
| Layer containers | `nn.ModuleList` or `nn.Sequential`, never a plain list |
| Training loop | zero_grad, forward, loss, backward, step |
| Saving | `state_dict`, not the whole object |

**Next:** Module 3, feeding real data into this loop with `Dataset` and `DataLoader`.